In [ ]:
# ============================================================================
# CELL 1: Setup Google Colab Environment
# ============================================================================
from google.colab import drive
drive.mount('/content/drive')
print("✓ Setup complete!")

Mounted at /content/drive
✓ Setup complete!


In [ ]:
!pip install -q openai tqdm pyarrow

In [ ]:
# ============================================================================
# CELL 2: Imports & Configuration
# ============================================================================
import pandas as pd
import numpy as np
import json, time, os, re, hashlib
from datetime import datetime
from typing import Dict, List, Optional
from tqdm.auto import tqdm
from openai import OpenAI
import openai

# ---- PATHS (update if needed) ----
BASE = "/content/drive/MyDrive/PropInsight"

INPUT_FILE   = f"{BASE}/preprocess/govenrment_websites/gov_websites_all_cleaned.csv"
OUTPUT_FILE  = f"{BASE}/labeled/government/gov_websites_labeled.csv"

# Checkpoints
CHECKPOINT_FILE = f"{BASE}/labeled/checkpoint_gov/labeling_checkpoint.csv"
HASH_PATH       = f"{BASE}/labeled/checkpoint_gov/gov_hashes.parquet"

# Ensure dirs exist
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
os.makedirs(os.path.dirname(CHECKPOINT_FILE), exist_ok=True)
os.makedirs(os.path.dirname(HASH_PATH), exist_ok=True)

# Optional filter (e.g., "HDB", "MAS", "BCA", "MND", "SFA", "SLA")
AGENCY_ONLY = ""  # set to "HDB" to label only HDB rows, else keep "" for all

# OpenAI Settings
OPENAI_MODEL      = "gpt-4o"
MAX_RETRIES       = 3
RETRY_DELAY       = 2
BATCH_SIZE        = 10   # checkpoint every N items
RATE_LIMIT_DELAY  = 0.8  # seconds between calls

print("✓ Configuration loaded")


✓ Configuration loaded


In [ ]:
# ============================================================================
# CELL 3: API Key
# ============================================================================
from google.colab import userdata

try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
    client = OpenAI(api_key=OPENAI_API_KEY)
    client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": "ping"}],
        max_tokens=4
    )
    print(f"✓ OpenAI client initialized — model: {OPENAI_MODEL}")
except Exception as e:
    print("✗ Could not initialize OpenAI client.")
    print("Detail:", e)
    raise


✓ OpenAI client initialized — model: gpt-4o


In [ ]:


# ============================================================================
# CELL 4: Helpers (prompt, API call, etc.)
# ============================================================================

def create_llm_prompt_gov(body: str,
                          agency: str,
                          rx_flags: List[str],
                          entity_labels: List[str]) -> str:
    """
    Gov-specific prompt: summarizes context for a press release / policy page.
    """
    rx_str     = ", ".join(sorted(set(rx_flags))) if rx_flags else "None"
    ent_str    = ", ".join(sorted(set(entity_labels))) if entity_labels else "None"
    agency_str = agency or "Unknown"

    return f"""Analyze this Singapore government article/press release and extract policy-aware sentiment and context.

AGENCY: {agency_str}
Regex policy flags: {rx_str}
Domain entities (labels only): {ent_str}

TEXT (truncated if long):
{body[:3000]}

Return ONLY valid JSON with these exact keys:
{{
  "overall_sentiment": "positive|neutral|negative",
  "policy_sentiment": "positive|neutral|negative",
  "price_sentiment": "rising|neutral|falling",
  "affordability_sentiment": "positive|neutral|negative",
  "policy_mentioned": "comma-separated policy terms or 'none'",
  "location": "planning area / district if any else 'none'",
  "emotion": "joy|anger|fear|trust|anticipation|surprise|sadness|disgust|neutral"
}}"""

def call_openai_api(client: OpenAI, prompt: str) -> Dict:
    """
    Jittered exponential-ish backoff within MAX_RETRIES.
    """
    delay = 1.0
    for attempt in range(MAX_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {"role":"system","content":"You are an expert on Singapore housing policy and macroprudential communications. Output valid JSON only."},
                    {"role":"user","content": prompt}
                ],
                temperature=0.3,
                max_tokens=450,
                response_format={"type":"json_object"}
            )
            return json.loads(resp.choices[0].message.content)
        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                print(f"✗ OpenAI error (final): {e}")
                break
            sleep = delay + float(np.random.uniform(0, 0.5))
            print(f"⚠ OpenAI error: {e}. Retry in {sleep:.1f}s")
            time.sleep(sleep)
            delay = min(delay * 2.0, 20.0)

    # fallback
    return {
        "overall_sentiment":"neutral",
        "policy_sentiment":"neutral",
        "price_sentiment":"neutral",
        "affordability_sentiment":"neutral",
        "policy_mentioned":"none",
        "location":"none",
        "emotion":"neutral"
    }

def _clamp_set(val, allowed, default="neutral"):
    v = (str(val or "")).lower().strip()
    return v if v in allowed else default

def normalize_llm_output(d: Dict) -> Dict:
    d = dict(d or {})
    d["overall_sentiment"]       = _clamp_set(d.get("overall_sentiment"),       {"positive","neutral","negative"})
    d["policy_sentiment"]        = _clamp_set(d.get("policy_sentiment"),        {"positive","neutral","negative"})
    d["price_sentiment"]         = _clamp_set(d.get("price_sentiment"),         {"rising","neutral","falling"})
    d["affordability_sentiment"] = _clamp_set(d.get("affordability_sentiment"), {"positive","neutral","negative"})
    d["emotion"]                 = _clamp_set(d.get("emotion"),                 {"joy","anger","fear","trust","anticipation","surprise","sadness","disgust","neutral"})
    d["location"]         = (d.get("location") or "none").strip()
    d["policy_mentioned"] = (d.get("policy_mentioned") or "none").strip()
    return d

def process_one_row(row: pd.Series, client: OpenAI) -> Dict:
    """
    Build context, create prompt, call model, normalize.
    """
    body   = str(row.get("body", "") or row.get("clean_text",""))
    agency = str(row.get("agency", ""))

    # rx_* columns to pass as hints
    rx_cols = [c for c in row.index if c.startswith("rx_")]
    rx_hits = [c.replace("rx_","") for c in rx_cols if bool(row.get(c, False))]

    # domain entity labels (from EntityRuler prepass)
    ents = row.get("entities", [])
    labset = []
    if isinstance(ents, list):
        for e in ents:
            lab = (e.get("label") if isinstance(e, dict) else None) or None
            if lab: labset.append(str(lab))

    prompt = create_llm_prompt_gov(body=body, agency=agency, rx_flags=rx_hits, entity_labels=labset)
    out    = call_openai_api(client, prompt)
    out    = normalize_llm_output(out)
    out["agency"] = agency
    return out

def is_boiler(s: str) -> bool:
    if not isinstance(s, str) or not s.strip(): return True
    s = s.lower()
    pats = ["skip to main", "share this page", "privacy policy", "terms of use", "prevnext", "site map"]
    return any(p in s for p in pats)

def _body_hash(s: str) -> str:
    s = (s or "").strip().lower()
    return hashlib.md5(s.encode("utf-8")).hexdigest()

In [ ]:

# ============================================================================
# CELL 5: Load data, prep columns, filter, checkpoint merge
# ============================================================================
print("Loading government websites data...")
df = pd.read_csv(INPUT_FILE)
print(f"✓ Loaded {len(df)} rows | columns={len(df.columns)}")

# optional agency filter
if AGENCY_ONLY:
    before = len(df)
    df = df[df.get("agency","").astype(str).str.upper()==AGENCY_ONLY.upper()].copy()
    print(f"[Filter] agency={AGENCY_ONLY} → kept {len(df)}/{before}")

# Ensure body exists
if "body" not in df.columns:
    title_col = "title" if "title" in df.columns else None
    text_col  = "clean_text" if "clean_text" in df.columns else None
    title_series = df[title_col].astype(str) if title_col else pd.Series([""]*len(df), index=df.index)
    text_series  = df[text_col].astype(str)  if text_col  else pd.Series([""]*len(df), index=df.index)
    df["body"] = (title_series + "\n\n" + text_series).str.strip()

# Prefilter low-signal
def low_signal(row) -> bool:
    txt = str(row.get("body",""))
    if len(txt) < 80: return True
    return is_boiler(txt)

pre = len(df)
df = df[~df.apply(low_signal, axis=1)].copy()
df.reset_index(drop=True, inplace=True)
print(f"[Prefilter] removed {pre-len(df)} → remaining {len(df)}")

# Create hash if missing
if "hash" not in df.columns:
    df["hash"] = df["body"].map(_body_hash)

# Load hash checkpoint
try:
    seen = pd.read_parquet(HASH_PATH).set_index("hash")
    _seen_set = set(seen.index)
except Exception:
    _seen_set = set()

print(f"New bodies to label (hash-check): {(~df['hash'].isin(_seen_set)).sum()} / {len(df)}")

# If checkpoint CSV exists, merge by hash so we can resume safely
start_idx = 0
if os.path.exists(CHECKPOINT_FILE):
    try:
        ckp = pd.read_csv(CHECKPOINT_FILE)
        if "hash" in ckp.columns:
            before_cols = set(df.columns)
            df = df.set_index("hash").combine_first(ckp.set_index("hash")).reset_index()
            # Align column order
            for col in set(df.columns)-before_cols:
                df[col] = df[col]
            start_idx = df["overall_sentiment"].notna().sum() if "overall_sentiment" in df.columns else 0
            print(f"✓ Merged checkpoint by hash. Resuming at row {start_idx}")
        else:
            print("⚠ Checkpoint has no 'hash'; skipping merge.")
    except Exception as e:
        print("⚠ Could not merge checkpoint:", e)

# Ensure label columns exist
label_columns = [
    'overall_sentiment','price_sentiment','policy_sentiment','affordability_sentiment',
    'location','policy_mentioned','emotion','agency'
]
for col in label_columns:
    if col not in df.columns:
        df[col] = None

print(f"\n✓ Ready to process {len(df) - start_idx} rows\n")

Loading government websites data...
✓ Loaded 59 rows | columns=121
[Prefilter] removed 0 → remaining 59
New bodies to label (hash-check): 59 / 59

✓ Ready to process 59 rows



In [ ]:
# ============================================================================
# CELL 6: Main labeling loop
# ============================================================================
pbar = tqdm(total=(len(df)-start_idx), desc="Labeling", unit="items")
processed_count, errors = 0, []

try:
    for idx in range(start_idx, len(df)):

        already_labeled = pd.notna(df.at[idx,'overall_sentiment'])
        already_seen    = df.at[idx,'hash'] in _seen_set

        if already_labeled or already_seen:
            pbar.update(1)
            continue

        try:
            labels = process_one_row(df.iloc[idx], client)
            for k, v in labels.items():
                df.at[idx, k] = v

            processed_count += 1
            pbar.update(1)

            # Save checkpoints periodically
            if processed_count % BATCH_SIZE == 0:
                df.to_csv(CHECKPOINT_FILE, index=False)
                # update hash store
                to_add = df.loc[df['overall_sentiment'].notna(), ['hash']].drop_duplicates()
                if len(to_add):
                    try:
                        cur = pd.read_parquet(HASH_PATH)
                    except Exception:
                        cur = pd.DataFrame(columns=["hash"])
                    merged = pd.concat([cur, to_add], ignore_index=True).drop_duplicates("hash")
                    merged.to_parquet(HASH_PATH, index=False)
                    _seen_set.update(set(to_add['hash']))

            time.sleep(RATE_LIMIT_DELAY)

        except KeyboardInterrupt:
            print("\nStopping… saving checkpoint.")
            df.to_csv(CHECKPOINT_FILE, index=False)
            raise

        except Exception as e:
            msg = f"Row {idx}: {e}"
            errors.append(msg)
            print("⚠", msg)
            continue

    pbar.close()

except KeyboardInterrupt:
    pbar.close()

# Final hash save
try:
    cur = pd.read_parquet(HASH_PATH)
except Exception:
    cur = pd.DataFrame(columns=["hash"])
to_add = df.loc[df['overall_sentiment'].notna(), ['hash']].drop_duplicates()
merged = pd.concat([cur, to_add], ignore_index=True).drop_duplicates("hash")
merged.to_parquet(HASH_PATH, index=False)
_seen_set.update(set(to_add['hash']))

print("\nDone. Processed:", processed_count, "| Errors:", len(errors))


Labeling:   0%|          | 0/59 [00:00<?, ?items/s]


Done. Processed: 59 | Errors: 0


In [ ]:
# ============================================================================
# CELL 7: Save & quick stats
# ============================================================================
try:
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"✓ Saved labeled dataset → {OUTPUT_FILE} | rows={len(df)}")
except Exception as e:
    print("✗ Error saving:", e)
    print("Checkpoint at:", CHECKPOINT_FILE)

print("\n=== SUMMARY ===")
def _vc(col):
    if col in df.columns: print(df[col].value_counts(dropna=True))
    else: print("(missing)")

print("\n📊 Overall Sentiment:")
_vc("overall_sentiment")

print("\n🏛 Policy Sentiment:")
_vc("policy_sentiment")

print("\n📈 Price Sentiment:")
_vc("price_sentiment")

print("\n💰 Affordability Sentiment:")
_vc("affordability_sentiment")

print("\n😊 Emotion:")
_vc("emotion")

print("\nTop policy terms (model output):")
if "policy_mentioned" in df.columns:
    pm = df[df["policy_mentioned"].notna() & (df["policy_mentioned"].str.lower()!="none")]["policy_mentioned"]
    if len(pm):
        print(pm.str.split(",").explode().str.strip().value_counts().head(20))
    else:
        print("(none)")
else:
    print("(missing)")


✓ Saved labeled dataset → /content/drive/MyDrive/PropInsight/labeled/government/gov_websites_labeled.csv | rows=59

=== SUMMARY ===

📊 Overall Sentiment:
overall_sentiment
neutral     36
positive    21
negative     2
Name: count, dtype: int64

🏛 Policy Sentiment:
policy_sentiment
positive    30
neutral     28
negative     1
Name: count, dtype: int64

📈 Price Sentiment:
price_sentiment
neutral    54
rising      4
falling     1
Name: count, dtype: int64

💰 Affordability Sentiment:
affordability_sentiment
neutral     53
positive     4
negative     2
Name: count, dtype: int64

😊 Emotion:
emotion
neutral         24
trust           16
anticipation    15
joy              2
disgust          2
Name: count, dtype: int64

Top policy terms (model output):
policy_mentioned
Built Environment Industry Transformation Map             5
food safety                                               5
Total Debt Servicing Ratio                                3
BuildSG                                          